In [ ]:
import json
import os
import shutil
import random
from pathlib import Path

old_label_map = {
    '자전거 도로(바닥표시)': 0,  
    '12.맨홀': 1,
    '맨홀':    1,
    '23.방치물(전동킥보드)': 2,
    '방치물(전동킥보드)':    2,
}


old_label_map = {
    '23.방치물(전동킥보드)': 0,  
    '방치물(전동킥보드)':    0,  
    '12.맨홀':              1,  
    '맨홀':                 1,  
}


dataset_pairs = [
    (r"C:\Users\konyang\Dataset\kickboard_image",  r"C:\Users\konyang\Dataset\kickboard_label",  'old'),
    (r"C:\Users\konyang\Dataset\manhole_image",    r"C:\Users\konyang\Dataset\manhole_label",    'old'),
    (r"C:\Users\konyang\Dataset\기둥 이미지",        r"C:\Users\konyang\Dataset\기둥 라벨링",        'new'),
    (r"C:\Users\konyang\Dataset\소화전 이미지",      r"C:\Users\konyang\Dataset\소화전 라벨링",      'new'),
    (r"C:\Users\konyang\Dataset\점자블록 이미지",    r"C:\Users\konyang\Dataset\점자블록 라벨링",    'new'),
]


output_dir = r"C:\Users\konyang\Dataset\detect_dataset"
for split in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)


random.seed(42)
converted = 0
skipped = 0

for img_dir, lbl_dir, fmt in dataset_pairs:
    json_files = [f for f in os.listdir(lbl_dir) if f.endswith('.json')]

    for fname in json_files:
        with open(os.path.join(lbl_dir, fname), 'r', encoding='utf-8') as f:
            data = json.load(f)

        lines = []

        
        if fmt == 'old':
            img_info   = data['images']
            img_width  = img_info['width']
            img_height = img_info['height']
            img_name   = img_info['filename']

            for ann in data['annotation']:
                label = ann['label']
                if label not in old_label_map:
                    continue
                class_id = old_label_map[label]

                if 'segmentation' in ann:
                    pts = ann['segmentation']
                    xs = [p['x'] for p in pts]
                    ys = [p['y'] for p in pts]
                    x_min, x_max = min(xs), max(xs)
                    y_min, y_max = min(ys), max(ys)
                elif 'bbox' in ann:
                    x_min, y_min, bw, bh = ann['bbox']
                    x_max = x_min + bw
                    y_max = y_min + bh
                else:
                    continue

                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width    = (x_max - x_min) / img_width
                height   = (y_max - y_min) / img_height

                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                width    = max(0, min(1, width))
                height   = max(0, min(1, height))

                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        
        elif fmt == 'new':
            img_info   = data['info']
            img_width  = img_info['width']
            img_height = img_info['height']
            img_name   = img_info['filename']

            # assign class based on folder
            if '기둥' in lbl_dir:
                class_id = 2  # bollard
            elif '소화전' in lbl_dir:
                class_id = 3  # fire hydrant
            elif '점자블록' in lbl_dir:
                class_id = 4  # tactile paving
            else:
                continue

            for ann in data['annotations']:
                bbox = ann['annotation_info'][0]  # [x_min, y_min, x_max, y_max]
                x_min, y_min, x_max, y_max = bbox

                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width    = (x_max - x_min) / img_width
                height   = (y_max - y_min) / img_height

                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                width    = max(0, min(1, width))
                height   = max(0, min(1, height))

                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        if not lines:
            skipped += 1
            continue

        # 80/20 train/val split
        split = 'train' if random.random() < 0.8 else 'val'

        # Copy image
        src_img = os.path.join(img_dir, img_name)
        dst_img = os.path.join(output_dir, f'images/{split}', img_name)
        if not os.path.exists(src_img):
            skipped += 1
            continue
        shutil.copy(src_img, dst_img)

        # Write label txt
        txt_name = Path(img_name).stem + '.txt'
        dst_lbl  = os.path.join(output_dir, f'labels/{split}', txt_name)
        with open(dst_lbl, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))

        converted += 1


In [ ]:
yaml_content = """path: C:/Users/konyang/Dataset/detect_dataset
train: images/train
val: images/val

nc: 5
names: ['kickboard', 'manhole', 'bollard', 'fire_hydrant', 'tactile_paving']
"""

with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("data.yaml created!")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

model.train(
    data=r'C:\Users\konyang\Dataset\detect_dataset\data.yaml',
    epochs=70,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,  
)